# Mixed-Frequency VaR with Kupiec and Christoffersen Backtests

**Module 06 — Financial Applications**

**Learning objectives:**
- Implement daily VaR from MIDAS-RV monthly volatility forecast
- Apply Kupiec (1995) unconditional coverage test
- Apply Christoffersen (1998) conditional coverage test
- Compare MIDAS-VaR against Historical Simulation and GARCH-VaR
- Interpret Basel III traffic light results

**Estimated time**: 13 minutes  
**Data**: S&P 500 daily returns (local CSV)

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

In [ ]:
section_divider("1. Load S&P 500 Daily Returns")

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load S&P 500 Daily Returns

In [ ]:
sp500 = pd.read_csv('../resources/sp500_returns.csv', parse_dates=['date'])
sp500 = sp500.set_index('date').sort_index()
daily_ret = sp500['return']

print(f"S&P 500 daily returns: {len(daily_ret)} observations")
print(f"Range: {daily_ret.index[0].strftime('%Y-%m-%d')} to {daily_ret.index[-1].strftime('%Y-%m-%d')}")
print(f"Mean daily return: {daily_ret.mean():.4f}")
print(f"Daily vol: {daily_ret.std():.4f} ({daily_ret.std()*np.sqrt(252):.1%} annualised)")

In [ ]:
section_divider("2. Compute Monthly MIDAS-RV Volatility Forecast")

## 2. Compute Monthly MIDAS-RV Volatility Forecast

Use a rolling MIDAS-RV estimate to produce monthly volatility forecasts. Then scale to daily VaR.

In [ ]:
def beta_weights(K, theta1, theta2):
    x = np.linspace(0.001, 0.999, K)
    unnorm = x**(theta1 - 1) * (1 - x)**(theta2 - 1)
    return unnorm / unnorm.sum()


def compute_monthly_rv(daily_ret):
    """Sum of squared daily returns per calendar month."""
    return (daily_ret**2).resample('MS').sum()


def fit_midas_rv_simple(rv_daily_arr, rv_monthly_arr, K=22):
    """Quick NLS fit for MIDAS-RV."""
    rv_d_log = np.log(rv_daily_arr + 1e-10)
    rv_m_log = np.log(rv_monthly_arr + 1e-10)
    T = len(rv_monthly_arr)
    
    def loss(params):
        mu, phi, t1, t2 = params
        if t1 <= 0 or t2 <= 0:
            return 1e10
        w = beta_weights(K, t1, t2)
        sse = 0
        for t in range(T):
            end_i = (t + 1) * K
            start_i = end_i - K
            if start_i < 0 or end_i > len(rv_d_log):
                continue
            lags = rv_d_log[start_i:end_i][::-1]
            fitted = mu + phi * (w @ lags)
            sse += (rv_m_log[t] - fitted)**2
        return sse
    
    res = minimize(loss, [rv_m_log.mean(), 0.5, 1.0, 5.0],
                   method='L-BFGS-B',
                   bounds=[(None,None), (0,2), (0.01,20), (0.01,20)],
                   options={'maxiter': 300})
    return res.x


# Compute monthly RV
rv_monthly = compute_monthly_rv(daily_ret)
rv_monthly = rv_monthly[rv_monthly.index >= pd.Timestamp('2005-01-01')]

# Fit MIDAS-RV on first 60% of sample for parameter estimation
T_m = len(rv_monthly)
train_m = int(T_m * 0.6)
K = 22

# Build daily RV array aligned with monthly
daily_sq = daily_ret**2
rv_d_arr = []
for m_date in rv_monthly.index[:train_m]:
    avail = daily_sq[daily_sq.index < m_date].tail(K).values
    if len(avail) == K:
        rv_d_arr.append(avail)

if rv_d_arr:
    rv_d_mat = np.array(rv_d_arr)
    rv_m_train = rv_monthly.iloc[:len(rv_d_arr)].values
    params = fit_midas_rv_simple(rv_d_mat.ravel(), rv_m_train, K)
    mu_rv, phi_rv, theta1_rv, theta2_rv = params
    w_rv = beta_weights(K, theta1_rv, theta2_rv)
    print(f"MIDAS-RV fitted: θ₁={theta1_rv:.3f}, θ₂={theta2_rv:.3f}, φ={phi_rv:.4f}")
else:
    # Fallback parameters
    mu_rv, phi_rv, theta1_rv, theta2_rv = -8.5, 0.6, 1.0, 5.0
    w_rv = beta_weights(K, theta1_rv, theta2_rv)
    print(f"Using default parameters.")

In [ ]:
section_divider("3. Generate VaR Estimates")

## 3. Generate VaR Estimates

Three VaR approaches compared:
1. **MIDAS-VaR**: Monthly MIDAS-RV forecast → scaled to daily
2. **Historical Simulation**: 250-day rolling empirical quantile
3. **GARCH(1,1)-VaR**: Simple GARCH daily variance

In [ ]:
alpha_var = 0.01  # 99% VaR
z_alpha = stats.norm.ppf(alpha_var)  # ≈ -2.326
window_hs = 250  # Historical simulation window

ret_arr = daily_ret.values
dates_arr = daily_ret.index
T_d = len(ret_arr)

var_midas = np.full(T_d, np.nan)
var_hs = np.full(T_d, np.nan)
var_garch = np.full(T_d, np.nan)

# Simple GARCH(1,1) recursion
garch_sigma2 = np.full(T_d, np.nan)
omega, alpha_g, beta_g = 0.00001, 0.09, 0.86  # Typical S&P 500 params
garch_sigma2[0] = np.var(ret_arr[:50])
for t in range(1, T_d):
    garch_sigma2[t] = omega + alpha_g * ret_arr[t-1]**2 + beta_g * garch_sigma2[t-1]

# Monthly MIDAS-RV → daily VaR
# Map each daily date to the preceding month's MIDAS-RV forecast
for t in range(window_hs, T_d):
    date = dates_arr[t]
    
    # Historical Simulation
    window = ret_arr[t-window_hs:t]
    var_hs[t] = -np.percentile(window, alpha_var * 100)  # Positive = loss limit
    
    # GARCH VaR
    var_garch[t] = -z_alpha * np.sqrt(garch_sigma2[t])
    
    # MIDAS-VaR: find the most recent month-end before this date
    month_start = date.to_period('M').to_timestamp()
    prev_month = month_start - pd.DateOffset(months=1)
    
    if prev_month in rv_monthly.index:
        # Get K daily squared returns before prev_month
        d_avail = daily_sq[daily_sq.index < prev_month].tail(K)
        if len(d_avail) == K:
            log_d = np.log(d_avail.values + 1e-10)[::-1]  # Most recent first
            log_rv_forecast = mu_rv + phi_rv * (w_rv @ log_d)
            daily_var_forecast = np.exp(log_rv_forecast) / 22.0  # Scale to daily
            var_midas[t] = -z_alpha * np.sqrt(daily_var_forecast)

# Use eval period: last 30% of sample
eval_start_d = int(T_d * 0.7)
ret_eval = ret_arr[eval_start_d:]
var_midas_eval = var_midas[eval_start_d:]
var_hs_eval = var_hs[eval_start_d:]
var_garch_eval = var_garch[eval_start_d:]

print(f"Evaluation period: {dates_arr[eval_start_d].strftime('%Y-%m-%d')} to {dates_arr[-1].strftime('%Y-%m-%d')}")
print(f"Number of days: {len(ret_eval)}")
print(f"Expected violations at 1% VaR: {len(ret_eval) * alpha_var:.0f}")

In [ ]:
section_divider("4. Backtesting: Kupiec and Christoffersen Tests")

## 4. Backtesting: Kupiec and Christoffersen Tests

In [ ]:
def kupiec_test(returns, var_estimates, alpha=0.01):
    """
    Kupiec (1995) unconditional coverage test.
    H0: violation rate = alpha.
    """
    valid = ~np.isnan(var_estimates)
    r = returns[valid]
    v = var_estimates[valid]
    
    violations = r < -v  # Return below negative VaR threshold
    T = len(violations)
    V = violations.sum()
    p_hat = V / T
    
    if V == 0:
        return p_hat, V, T, np.nan, np.nan, 'No violations'
    if V == T:
        return p_hat, V, T, np.nan, np.nan, 'All violations'
    
    lr_uc = 2 * (
        V * np.log(p_hat / alpha) +
        (T - V) * np.log((1 - p_hat) / (1 - alpha))
    )
    pval = stats.chi2.sf(lr_uc, df=1)
    
    if pval >= 0.05:
        result = 'PASS (green)'
    elif p_hat > alpha:
        result = 'FAIL — too many violations (amber/red)'
    else:
        result = 'FAIL — too few violations (over-conservative)'
    
    return p_hat, V, T, lr_uc, pval, result


def christoffersen_test(returns, var_estimates):
    """
    Christoffersen (1998) independence test.
    H0: violation indicators are serially independent.
    """
    valid = ~np.isnan(var_estimates)
    r = returns[valid]
    v = var_estimates[valid]
    
    viol = (r < -v).astype(int)
    T = len(viol)
    
    # Transition counts
    n00 = np.sum((viol[:-1] == 0) & (viol[1:] == 0))
    n01 = np.sum((viol[:-1] == 0) & (viol[1:] == 1))
    n10 = np.sum((viol[:-1] == 1) & (viol[1:] == 0))
    n11 = np.sum((viol[:-1] == 1) & (viol[1:] == 1))
    
    denom01 = max(n00 + n01, 1)
    denom11 = max(n10 + n11, 1)
    pi01 = n01 / denom01
    pi11 = n11 / denom11
    pi = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    
    eps = 1e-10
    lr_ind = 2 * (
        n00 * np.log(1 - pi01 + eps) + n01 * np.log(pi01 + eps) +
        n10 * np.log(1 - pi11 + eps) + n11 * np.log(pi11 + eps) -
        (n00 + n10) * np.log(1 - pi + eps) -
        (n01 + n11) * np.log(pi + eps)
    )
    
    pval = stats.chi2.sf(max(lr_ind, 0), df=1)
    clustering = pi11 > pi01  # Violations tend to cluster
    
    return lr_ind, pval, pi01, pi11, clustering


# Run backtests
models = {
    'MIDAS-VaR': var_midas_eval,
    'Historical Sim': var_hs_eval,
    'GARCH(1,1)-VaR': var_garch_eval,
}

print(f"=== VaR Backtest Results ({alpha_var*100:.0f}% VaR, 1% expected violation) ===")
print()
print(f"{'Model':<20} {'Viol. Rate':<12} {'Violations':<12} {'Kupiec p':<12} {'Result'}")
print("-" * 75)

for name, var_est in models.items():
    p_hat, V, T_v, lr_uc, pval, result = kupiec_test(ret_eval, var_est, alpha=alpha_var)
    print(f"{name:<20} {p_hat:<12.4f} {V}/{T_v:<12} {pval if pval is not None else 'nan':<12.4f} {result}")

print()
print(f"=== Christoffersen Independence Test ===")
print(f"{'Model':<20} {'π₀₁':<10} {'π₁₁':<10} {'Ind. p':<10} {'Clustering?'}")
print("-" * 65)

for name, var_est in models.items():
    lr_ind, pval_ind, pi01, pi11, clustering = christoffersen_test(ret_eval, var_est)
    cluster_str = 'YES (violations cluster)' if clustering else 'No clustering'
    print(f"{name:<20} {pi01:<10.4f} {pi11:<10.4f} {pval_ind:<10.4f} {cluster_str}")

In [ ]:
section_divider("5. Basel III Traffic Light Assessment")

## 5. Basel III Traffic Light Assessment

In [ ]:
# Basel III: 250-day rolling backtest
window_basel = 250
T_eval = len(ret_eval)

violations_midas = (ret_eval < -var_midas_eval) & ~np.isnan(var_midas_eval)
violations_hs = (ret_eval < -var_hs_eval) & ~np.isnan(var_hs_eval)
violations_garch = (ret_eval < -var_garch_eval) & ~np.isnan(var_garch_eval)

def basel_traffic_light(n_violations):
    """Basel III traffic light based on violations in 250 days."""
    if n_violations <= 4:
        return 'GREEN (multiplier 3.0)'
    elif n_violations <= 9:
        return f'YELLOW (multiplier 3.{n_violations - 4})'
    else:
        return 'RED (multiplier 4.0)'

print("=== Basel III Traffic Light (250-day period) ===")
print(f"Expected violations at 1% VaR: {250 * alpha_var:.0f}")
print()

for name, viols in [('MIDAS-VaR', violations_midas), ('Historical Sim', violations_hs), ('GARCH(1,1)', violations_garch)]:
    if T_eval >= window_basel:
        recent_viols = viols[-window_basel:].sum()
    else:
        recent_viols = viols.sum()
    traffic = basel_traffic_light(recent_viols)
    print(f"{name:<20}: {recent_viols} violations → {traffic}")

# Plot violations timeline
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
eval_dates_plot = daily_ret.index[eval_start_d:]

for i, (name, var_est, viols) in enumerate([
    ('MIDAS-VaR', var_midas_eval, violations_midas),
    ('Historical Sim', var_hs_eval, violations_hs),
    ('GARCH(1,1)-VaR', var_garch_eval, violations_garch),
]):
    ax = axes[i]
    ax.plot(eval_dates_plot, ret_eval, 'gray', linewidth=0.5, alpha=0.6, label='Returns')
    ax.plot(eval_dates_plot, -var_est, 'b-', linewidth=1.0, label=f'{name}')
    viol_dates = eval_dates_plot[viols]
    viol_vals = ret_eval[viols]
    ax.scatter(viol_dates, viol_vals, color='red', s=15, zorder=5, label=f'Violations ({viols.sum()})')
    ax.set_title(name)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.suptitle('Daily Returns vs VaR Estimates (Red = Violation)', y=1.01)
plt.tight_layout()
plt.savefig('../resources/var_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

This notebook implemented the full mixed-frequency VaR pipeline:

1. **MIDAS-RV → daily VaR**: Monthly MIDAS-RV forecast scaled by $1/\sqrt{22}$ to daily standard deviation, then multiplied by normal quantile $z_{0.01} = -2.326$
2. **Kupiec test**: Checks if violation rate equals 1%. Pass = model is well-calibrated.
3. **Christoffersen test**: Checks if violations cluster. Clustering ($\pi_{11} > \pi_{01}$) indicates the model fails to adapt to volatility regimes.
4. **Basel III traffic light**: Green (≤4 violations per 250 days) → no capital add-on; Red (≥10) → 33% capital increase.

**Key finding**: MIDAS-VaR typically shows fewer clustered violations than Historical Simulation (which is slow to update) while being more stable than pure GARCH-VaR (which over-reacts to recent shocks).

**Next**: `../exercises/01_financial_self_check.py` — self-check exercise.

In [ ]:
key_takeaways(["MIDAS-RV → daily VaR", "Kupiec test", "Christoffersen test", "Basel III traffic light"])